# Account Changes Batch — Unit / Invariant Tests

Standalone tests for the cleaning functions in `account_changes_batch_cleaning_v4.ipynb`. They run entirely on tiny in-memory fixtures — **no S3 access and no full data read required**.

> The cleaning functions below are **copied** from the cleaning notebook so this notebook runs on its own. If you change a function there, update it here too (or, longer term, move the functions into a shared `.py` module and import them in both notebooks).

In [ ]:
%%pyspark project.spark.compatibility
from pyspark.sql import SparkSession, DataFrame, Row
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType
import contextlib
import hashlib
import re

In [ ]:
%%pyspark project.spark.compatibility
spark = SparkSession.builder.appName("account_changes_batch_tests").getOrCreate()

## Cleaning functions (copied from the cleaning notebook — keep in sync)

In [ ]:
%%pyspark project.spark.compatibility
def flatten_json_column_with_schema(
    df: DataFrame,
    json_col: str,
    schema: StructType,
    prefix: str = None,
    drop_original: bool = False,
) -> DataFrame:
    """Parse a JSON-string column with a given schema and promote each field to its own column."""
    parsed = F.from_json(F.col(json_col), schema)
    for field in schema.fieldNames():
        output_col = f"{prefix}_{field}" if prefix else field
        df = df.withColumn(output_col, parsed[field])
    if drop_original:
        df = df.drop(json_col)
    return df

In [ ]:
%%pyspark project.spark.compatibility
notes_schema = StructType([
    StructField("mno", StringType(), True),
    StructField("subId", StringType(), True),
    StructField("msisdnChangeSide", StringType(), True),
    StructField("correlationId", StringType(), True),
    StructField("otherMSISDN", StringType(), True),
])


In [ ]:
%%pyspark project.spark.compatibility
def is_blank(c):
    """Return a boolean column that is True when `c` is null, literal "\\N", empty, or whitespace-only."""
    t = F.trim(F.col(c))
    return F.col(c).isNull() | (t == "\\N") | (t == "")

In [ ]:
%%pyspark project.spark.compatibility
def derive_mno(df):
    """
    Derive `mno` from fields parsed out of `notes`. Output domain: TELUS / BELL / ROGERS.

    Mapping (Simon's rules + Helena's correction that buckets 5-8 are Bell, not Rogers):
      mno = TELUS                                                              -> TELUS
      mno = BELL_LANDLINE                                                      -> BELL
      mno blank + any of subId / correlationId / msisdnChangeSide / otherMSISDN -> BELL
      everything else blank (notes is empty / \\N)                              -> ROGERS
    """
    return df.withColumn(
        "mno",
        F.when(F.upper(F.col("mno")) == "TELUS", "TELUS")
         .when(F.upper(F.col("mno")) == "BELL_LANDLINE", "BELL")
         .when(
             is_blank("mno")
             & (
                 ~is_blank("subId")
                 | ~is_blank("correlationId")
                 | ~is_blank("msisdnChangeSide")
                 | ~is_blank("otherMSISDN")
             ),
             "BELL",
         )
         .when(is_blank("mno"), "ROGERS")
         .otherwise(F.col("mno")),
    )

In [ ]:
%%pyspark project.spark.compatibility
def standardize_timestamp(df, ts_col, output_col=None):
    """Parse a string timestamp (yyyy-MM-dd HH:mm:ss or yyyy-MM-dd) into TimestampType; unparseable -> null. Wall-clock time is interpreted in the session timezone (set to America/Toronto)."""
    target_col = output_col or ts_col
    parsed = F.coalesce(
        F.try_to_timestamp(F.col(ts_col), F.lit("yyyy-MM-dd HH:mm:ss")),
        F.try_to_timestamp(F.col(ts_col), F.lit("yyyy-MM-dd")),
    )
    return df.withColumn(target_col, parsed)

In [ ]:
%%pyspark project.spark.compatibility
# Project bronze -> §6.1.2 silver schema (13 columns)

def clean_str(c):
    """Trim; convert literal "\\N" and empty string to null."""
    v = F.trim(F.col(c))
    return F.when((v == "\\N") | (v == ""), None).otherwise(v)

def safe_long(c):
    """Cast a string column to bigint. Keep only pure-digit strings; non-numeric -> null.
    LongType (bigint) is required: bronze `id` is bigint and values routinely exceed
    Int32 max (~2.147B) -- IntegerType would silently null ~85% of rows on overflow."""
    s = clean_str(c)
    return F.when(s.rlike("^[0-9]+$"), s).otherwise(None).cast(LongType())

def to_silver(df):
    """Project the bronze frame onto the §6.1.2 silver schema (13 columns).
    The four extras (subId / msisdnChangeSide / correlationId / otherMSISDN) are kept:
    not analyzed in phase 1 but used by derive_mno upstream and likely needed downstream."""
    # deterministic UUID (8-4-4-4-12, lowercase) derived from bronze id, for synthetic correlationId
    _h = F.sha2(F.col("id").cast("string"), 256)          # 64 lowercase hex
    _uuid_from_id = F.concat_ws(
        "-",
        F.substring(_h, 1, 8),
        F.substring(_h, 9, 4),
        F.substring(_h, 13, 4),
        F.substring(_h, 17, 4),
        F.substring(_h, 21, 12),
    )                                                     # 32 hex + 4 hyphens = 36 chars

    return df.select(
        F.sha2(F.col("id").cast("string"), 256).alias("record_id"),               # from scratch: sha256(id), STRING
        clean_str("change_key").alias("phone_number_AC_hash"),
        F.upper(clean_str("change_type")).alias("event_type"),
        F.col("timestamp").alias("event_timestamp"),
        F.col("mno"),
        clean_str("subId").alias("subId"),
        clean_str("msisdnChangeSide").alias("msisdnChangeSide"),
        F.coalesce(
            F.lower(clean_str("correlationId")),                                 # real value kept, lowercased
            F.concat(F.lit("gen-"), _uuid_from_id),                              # blank -> deterministic synthetic
        ).alias("correlationId"),
        clean_str("otherMSISDN").alias("otherMSISDN"),
        F.to_timestamp(F.lit("2026-03-16 00:00:00")).alias("ingestion_ts"),  # placeholder, to be replaced
        F.lit("MYSQL").alias("source_name"),
        safe_long("id").alias("source_event_id"),                            # provenance: raw bronze id
        F.lit(1).cast(IntegerType()).alias("schema_version"),
    )

## Test fixture

In [ ]:
%%pyspark project.spark.compatibility
# Fixture Silver frame, built from a tiny synthetic bronze sample through the SAME chain
# as production (flatten -> derive_mno -> standardize_timestamp -> to_silver -> dedupe).
# Used by test_extra_fields_kept and test_silver_schema_exact, which only inspect
# columns/schema (no row values), so a 2-row fixture is enough and needs no S3.
_bronze_fixture = spark.createDataFrame([
    Row(id="1", change_key="k1", change_type="SIM_CHANGE",
        timestamp="2023-11-26 00:00:00", notes='{"mno":"TELUS"}'),
    Row(id="2", change_key="k2", change_type="DEVICE_CHANGE",
        timestamp="2023-07-15 12:00:00", notes="\\N"),
])
_p = flatten_json_column_with_schema(_bronze_fixture, "notes", notes_schema)
_p = derive_mno(_p)
_p = standardize_timestamp(_p, "timestamp")
acb_final = to_silver(_p).dropDuplicates()

## Unit / invariant tests

In [ ]:
%%pyspark project.spark.compatibility
# ============================================================
# Unit / Invariant tests - exercise the cleaning functions with fixtures
# Run inside the notebook, or copy this block into test_account_changes_batch.py
# under pytest (would need a module-scope `spark` fixture).
# ============================================================

def _df(rows):
    """Build a tiny DataFrame from a list of Row objects (test fixture helper)."""
    return spark.createDataFrame(rows)

@contextlib.contextmanager
def _toronto_session():
    """Pin session.timeZone to America/Toronto for the duration of a test (timestamp determinism)."""
    prev = spark.conf.get("spark.sql.session.timeZone")
    spark.conf.set("spark.sql.session.timeZone", "America/Toronto")
    try:
        yield
    finally:
        spark.conf.set("spark.sql.session.timeZone", prev)

def _ts_str(df, col_name):
    """Format a timestamp column to string via Spark (session tz) and collect, avoiding
    PySpark's collect()-time tz conversion to the JVM default."""
    return [r[0] for r in df.select(
        F.date_format(F.col(col_name), "yyyy-MM-dd HH:mm:ss")
    ).collect()]

# ---------- JSON expand ----------
def test_json_expand_five_variants():
    rows = [
        Row(notes='{"mno":"TELUS"}'),
        Row(notes='{"mno":"BELL_LANDLINE"}'),
        Row(notes='{"subId":"s1"}'),
        Row(notes='{"correlationId":"c1"}'),
        Row(notes="\\N"),
    ]
    o = flatten_json_column_with_schema(_df(rows), "notes", notes_schema).collect()
    assert o[0]["mno"] == "TELUS"
    assert o[1]["mno"] == "BELL_LANDLINE"
    assert o[2]["subId"] == "s1" and o[2]["mno"] is None
    assert o[3]["correlationId"] == "c1" and o[3]["mno"] is None
    assert o[4]["mno"] is None and o[4]["subId"] is None

def test_notes_null_empty_malformed():
    # notes = null / empty / malformed JSON / \N -> all parsed fields null (from_json PERMISSIVE)
    rows = [Row(notes=None), Row(notes=""), Row(notes='{"mno":'), Row(notes="\\N")]
    o = flatten_json_column_with_schema(_df(rows), "notes", notes_schema).collect()
    for r in o:
        assert r["mno"] is None and r["subId"] is None

def test_notes_extra_keys_ignored():
    # Unknown JSON keys are ignored; known keys parse normally
    rows = [Row(notes='{"mno":"TELUS","foo":"bar","subId":"s9"}')]
    o = flatten_json_column_with_schema(_df(rows), "notes", notes_schema).collect()[0]
    assert o["mno"] == "TELUS" and o["subId"] == "s9"

def test_notes_value_literal_backslash_n():
    # JSON value is the literal string "\N" -> parsed as the two-char string, not null
    rows = [Row(notes='{"mno":"\\\\N"}')]
    o = flatten_json_column_with_schema(_df(rows), "notes", notes_schema).collect()[0]
    assert o["mno"] == "\\N"

# ---------- per-MNO mapping (3 values: TELUS / BELL / ROGERS) ----------
def test_mno_mapping():
    rows = [
        Row(notes='{"mno":"telus"}'),                # case-insensitive
        Row(notes='{"mno":"BELL_LANDLINE"}'),         # landline       -> BELL
        Row(notes='{"subId":"s1"}'),                  # mobile         -> BELL
        Row(notes='{"correlationId":"c1"}'),          # other id field -> BELL
        Row(notes="\\N"),                             # everything null -> ROGERS
    ]
    parsed = flatten_json_column_with_schema(_df(rows), "notes", notes_schema)
    got = [r["mno"] for r in derive_mno(parsed).collect()]
    assert got == ["TELUS", "BELL", "BELL", "BELL", "ROGERS"]

def test_mno_only_three_values():
    # Invariant: for known inputs, derive_mno output domain is a subset of {TELUS, BELL, ROGERS}
    rows = [
        Row(notes='{"mno":"TELUS"}'),
        Row(notes='{"mno":"BELL_LANDLINE"}'),
        Row(notes='{"subId":"s1"}'),
        Row(notes='{"msisdnChangeSide":"A"}'),
        Row(notes='{"otherMSISDN":"123"}'),
        Row(notes='{"correlationId":"c1","subId":"s1"}'),
        Row(notes="\\N"),
        Row(notes='{}'),
    ]
    parsed = flatten_json_column_with_schema(_df(rows), "notes", notes_schema)
    got = {r["mno"] for r in derive_mno(parsed).collect()}
    assert got <= {"TELUS", "BELL", "ROGERS"}, got

def test_mno_precedence_mno_over_other_fields():
    # When `mno` has a value, it wins even if subId / correlationId are also present
    rows = [Row(notes='{"mno":"TELUS","subId":"s1","correlationId":"c1"}')]
    parsed = flatten_json_column_with_schema(_df(rows), "notes", notes_schema)
    assert derive_mno(parsed).collect()[0]["mno"] == "TELUS"

def test_mno_unknown_value_passthrough():
    # Unknown mno value falls through the `otherwise` branch and is kept as-is
    rows = [Row(notes='{"mno":"FREEDOM"}')]
    parsed = flatten_json_column_with_schema(_df(rows), "notes", notes_schema)
    assert derive_mno(parsed).collect()[0]["mno"] == "FREEDOM"

def test_mno_blank_variants_all_rogers():
    # Different "everything is blank" inputs all bucket to ROGERS, including a literal "\N" value
    rows = [
        Row(notes=None),
        Row(notes="{}"),
        Row(notes="\\N"),
        Row(notes='{"correlationId":"\\\\N"}'),
    ]
    parsed = flatten_json_column_with_schema(_df(rows), "notes", notes_schema)
    got = [r["mno"] for r in derive_mno(parsed).collect()]
    assert got == ["ROGERS", "ROGERS", "ROGERS", "ROGERS"]

def test_mno_empty_string_treated_blank():
    # Empty-string fields are treated as blank: {"mno":""} -> ROGERS, {"subId":""} -> ROGERS
    rows = [Row(notes='{"mno":""}'), Row(notes='{"subId":""}')]
    parsed = flatten_json_column_with_schema(_df(rows), "notes", notes_schema)
    got = [r["mno"] for r in derive_mno(parsed).collect()]
    assert got == ["ROGERS", "ROGERS"]

# ---------- timestamp parsing (no timezone shift; bronze is already EST/EDT) ----------
def test_timestamp_preserves_wall_clock_winter():
    # November input is EST; wall clock is preserved as-is (no UTC conversion)
    with _toronto_session():
        out = _ts_str(
            standardize_timestamp(_df([Row(timestamp="2023-11-26 01:02:07")]), "timestamp"),
            "timestamp",
        )
        assert out == ["2023-11-26 01:02:07"]

def test_timestamp_preserves_wall_clock_summer():
    # July input is EDT; wall clock is preserved as-is (no DST shift either)
    with _toronto_session():
        out = _ts_str(
            standardize_timestamp(_df([Row(timestamp="2023-07-15 12:00:00")]), "timestamp"),
            "timestamp",
        )
        assert out == ["2023-07-15 12:00:00"]

def test_timestamp_date_only_and_invalid():
    # Date-only string parses to midnight; null / \N / garbage -> null
    with _toronto_session():
        out_df = standardize_timestamp(
            _df([Row(timestamp="2023-11-26"), Row(timestamp="\\N"),
                 Row(timestamp="garbage"), Row(timestamp=None)]),
            "timestamp",
        )
        strs = _ts_str(out_df, "timestamp")
        nulls = [r[0] for r in out_df.select(F.col("timestamp").isNull()).collect()]
        assert strs[0] == "2023-11-26 00:00:00"
        assert nulls[1] is True
        assert nulls[2] is True
        assert nulls[3] is True

# ---------- dedup ----------
def test_dedupe_collapses_duplicates():
    rows = [Row(id="1", change_key="k", change_type="SIM_CHANGE"),
            Row(id="1", change_key="k", change_type="SIM_CHANGE")]
    assert _df(rows).dropDuplicates().count() == 1

def test_dedupe_keeps_distinct_rows():
    # If any field differs, the row is kept
    rows = [Row(id="1", change_key="k", change_type="SIM_CHANGE"),
            Row(id="1", change_key="k", change_type="DEVICE_CHANGE")]
    assert _df(rows).dropDuplicates().count() == 2

# ---------- silver schema / types ----------
def test_extra_fields_kept():
    # The four "extras" are kept in silver (13-column schema); `notes` is parsed away
    extras = {"subId", "msisdnChangeSide", "correlationId", "otherMSISDN"}
    assert extras <= set(acb_final.columns)
    assert "notes" not in acb_final.columns

def test_silver_schema_exact():
    expected = [
        ("record_id", "string"),                    # sha256(bronze id) -> 64-char hex STRING
        ("phone_number_AC_hash", "string"),
        ("event_type", "string"),
        ("event_timestamp", "timestamp"),
        ("mno", "string"),
        ("subId", "string"),
        ("msisdnChangeSide", "string"),
        ("correlationId", "string"),
        ("otherMSISDN", "string"),
        ("ingestion_ts", "timestamp"),
        ("source_name", "string"),
        ("source_event_id", "bigint"),              # also bigint - stores bronze `id`
        ("schema_version", "int"),
    ]
    got = [(f.name, f.dataType.simpleString()) for f in acb_final.schema.fields]
    assert got == expected

def test_source_event_id_casting():
    # Numeric -> bigint; padded with spaces -> trimmed and cast; non-numeric / \N -> null.
    # Last fixture row uses a value above Int32 max (~2.147B) to verify Long handles it.
    rows = [
        Row(id="1611966072",  change_key="k", change_type="X", timestamp="2023-11-26", mno="ROGERS", subId="\\N", msisdnChangeSide="\\N", correlationId="\\N", otherMSISDN="\\N"),
        Row(id=" 42 ",         change_key="k", change_type="X", timestamp="2023-11-26", mno="ROGERS", subId="\\N", msisdnChangeSide="\\N", correlationId="\\N", otherMSISDN="\\N"),
        Row(id="abc",          change_key="k", change_type="X", timestamp="2023-11-26", mno="ROGERS", subId="\\N", msisdnChangeSide="\\N", correlationId="\\N", otherMSISDN="\\N"),
        Row(id="\\N",          change_key="k", change_type="X", timestamp="2023-11-26", mno="ROGERS", subId="\\N", msisdnChangeSide="\\N", correlationId="\\N", otherMSISDN="\\N"),
        Row(id="5128020852",   change_key="k", change_type="X", timestamp="2023-11-26", mno="ROGERS", subId="\\N", msisdnChangeSide="\\N", correlationId="\\N", otherMSISDN="\\N"),
    ]
    got = [r["source_event_id"] for r in to_silver(_df(rows)).collect()]
    assert got == [1611966072, 42, None, None, 5128020852]

def test_record_id_sha256_of_id():
    # record_id is a silver-owned surrogate = sha256(raw bronze id string): 64-char lowercase
    # hex, non-null for any non-null id, deterministic (matches a plain sha256), unique per id.
    rows = [
        Row(id="1611966072", change_key="k", change_type="X", timestamp="2023-11-26", mno="ROGERS",
            subId="\\N", msisdnChangeSide="\\N", correlationId="\\N", otherMSISDN="\\N"),
        Row(id="abc",        change_key="k", change_type="X", timestamp="2023-11-26", mno="ROGERS",
            subId="\\N", msisdnChangeSide="\\N", correlationId="\\N", otherMSISDN="\\N"),
    ]
    out = to_silver(_df(rows)).collect()
    for r in out:
        assert r["record_id"] is not None
        assert re.fullmatch(r"[0-9a-f]{64}", r["record_id"])
    assert out[0]["record_id"] == hashlib.sha256("1611966072".encode()).hexdigest()
    assert out[0]["record_id"] != out[1]["record_id"]   # distinct ids -> distinct record_id

def test_correlationid_real_lowercased_null_gen_filled():
    # Real correlationId kept (lowercased); blank -> deterministic "gen-" + 36-char UUID-shaped
    # value (strip "gen-" and it matches the real-value shape; prefix marks synthetic rows).
    rows = [
        Row(id="1", change_key="k", change_type="X", timestamp="2023-11-26", mno="BELL",
            subId="\\N", msisdnChangeSide="\\N",
            correlationId="B82C65A7-0F65-4AB7-980F-393AC089C9BB", otherMSISDN="\\N"),
        Row(id="2", change_key="k", change_type="X", timestamp="2023-11-26", mno="ROGERS",
            subId="\\N", msisdnChangeSide="\\N", correlationId="\\N", otherMSISDN="\\N"),
    ]
    out = {r["source_event_id"]: r["correlationId"] for r in to_silver(_df(rows)).collect()}
    assert out[1] == "b82c65a7-0f65-4ab7-980f-393ac089c9bb"           # real value, lowercased
    assert out[2].startswith("gen-")                                  # blank -> synthetic
    assert re.fullmatch(r"[0-9a-f]{8}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{4}-[0-9a-f]{12}", out[2][4:])

def test_constants_and_event_type_normalized():
    # Constant columns + event_type is uppercased and trimmed
    rows = [Row(id="1", change_key="k", change_type=" sim_change ",
                timestamp="2023-11-26", mno="ROGERS",
                subId="\\N", msisdnChangeSide="\\N", correlationId="\\N", otherMSISDN="\\N")]
    r = to_silver(_df(rows)).collect()[0]
    assert r["source_name"] == "MYSQL"
    assert r["schema_version"] == 1
    assert r["source_event_id"] == 1
    assert r["event_type"] == "SIM_CHANGE"

def test_event_type_hyphen_uppercase():
    # Hyphenated change_type values (STATUS_CHANGE-C / -AA / etc.) keep the hyphen after upper()
    rows = [Row(id="1", change_key="k", change_type="status_change-c",
                timestamp="2023-11-26", mno="ROGERS",
                subId="\\N", msisdnChangeSide="\\N", correlationId="\\N", otherMSISDN="\\N"),
            Row(id="2", change_key="k", change_type="STATUS_CHANGE-AA",
                timestamp="2023-11-26", mno="BELL",
                subId="\\N", msisdnChangeSide="\\N", correlationId="\\N", otherMSISDN="\\N")]
    got = [r["event_type"] for r in to_silver(_df(rows)).collect()]
    assert got == ["STATUS_CHANGE-C", "STATUS_CHANGE-AA"]

In [ ]:
%%pyspark project.spark.compatibility
# Run all unit tests
_tests = [
    test_json_expand_five_variants,
    test_notes_null_empty_malformed,
    test_notes_extra_keys_ignored,
    test_notes_value_literal_backslash_n,
    test_mno_mapping,
    test_mno_only_three_values,
    test_mno_precedence_mno_over_other_fields,
    test_mno_unknown_value_passthrough,
    test_mno_blank_variants_all_rogers,
    test_mno_empty_string_treated_blank,
    test_timestamp_preserves_wall_clock_winter,
    test_timestamp_preserves_wall_clock_summer,
    test_timestamp_date_only_and_invalid,
    test_dedupe_collapses_duplicates,
    test_dedupe_keeps_distinct_rows,
    test_extra_fields_kept,
    test_silver_schema_exact,
    test_source_event_id_casting,
    test_record_id_sha256_of_id,
    test_correlationid_real_lowercased_null_gen_filled,
    test_constants_and_event_type_normalized,
    test_event_type_hyphen_uppercase,
]
_failed = 0
for _t in _tests:
    try:
        _t(); print("PASS  " + _t.__name__)
    except AssertionError as e:
        _failed += 1; print("FAIL  " + _t.__name__ + ": " + str(e))
    except Exception as e:
        _failed += 1; print("ERROR " + _t.__name__ + ": " + type(e).__name__ + ": " + str(e))
print()
print(str(len(_tests) - _failed) + "/" + str(len(_tests)) + " passed")